# EDA Samuel — `impact_load`, `threat_rank`, dan `cycle_code`

**Pertanyaan:** Seberapa kuat hubungan antara `impact_load` dan `threat_rank`, dan apakah pola deskriptifnya terlihat serupa pada periode `cycle_code` yang berbeda?

Notebook ini memeriksa data regresi TP2 yang resmi, mengonfirmasi target lewat skema train–test–sample, lalu menghitung hubungan numerik dan ringkasan periode. Saat file regresi belum tersedia, notebook menampilkan status blocker dan nama file yang diperlukan. Ia tidak mengganti data regresi dengan classification.

**Batas interpretasi:** `cycle_code` diurutkan sebagai asumsi kerja untuk membentuk tiga periode berurutan. Makna domainnya perlu dikonfirmasi. Korelasi dan perbedaan distribusi bersifat deskriptif; tidak membuktikan sebab-akibat, drift, atau signifikansi statistik.

## tl;dr

**Status saat audit awal (30 Oktober 2026): `BLOCKED_MISSING_REGRESSION_DATA`.** Folder regresi TP2 belum berisi file resmi yang diperlukan. Hasil korelasi belum tersedia. Cell status berikutnya menghitung ulang status dari file yang saat ini ada.

## Context & Methods

- Unit analisis: satu baris train regresi.
- Target hanya dipastikan jika `threat_rank` adalah satu-satunya kolom train-only dan sample submission berisi tepat `record_id` serta `threat_rank`.
- Pearson merangkum asosiasi linear untuk pasangan numerik finite yang bervariasi; Spearman merangkum asosiasi peringkat. Hasilnya tetap deskriptif.
- Nama `threat_rank` belum mengonfirmasi skala atau makna domainnya; pengertian tersebut tetap UNKNOWN sampai definisi resmi tersedia.
- `impact_load` dibagi ke dalam delapan bin dengan batas sama yang dihitung sekali dari train dan dipakai bersama di semua periode. Ringkasan meliputi median dan kuartil target.
- Periode membagi nilai `cycle_code` terurut menjadi tiga kelompok berukuran hampir sama, tanpa memakai target untuk menentukan batas.
- Outlier univariat ditandai sebagai nilai di luar 1.5 × IQR. Flag ini bukan alasan otomatis menghapus observasi.

## Data

File yang dicari: `data/raw/regression/Train.csv`, `Test.csv`, dan `SampleSubmission.csv`. `TP2_DATA_DIR` dapat diarahkan ke direktori lain yang berisi folder `regression/`. Data mentah dibaca saja.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "utils" / "paths.py").is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "utils" / "paths.py").is_file():
    raise FileNotFoundError("Open this notebook from the TP2 repository or a subfolder thereof.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.samuel import run_eda

report = run_eda()
print("Status:", report["status"])
print("Regression data folder:", report["input_root"])
if report["status"].startswith("BLOCKED"):
    print("Required official files:", ", ".join(report.get("missing_files", [])))
    print(report.get("next_action", ""))
else:
    print("Confirmed target:", report["target_verification"]["confirmed_target"])
    print("Train/test shapes:", report["schema"]["train_shape"], report["schema"]["test_shape"])
    print("Cycle comparison:", report["temporal"]["status"])
    if report["temporal"]["status"] != "COMPLETE":
        print("Insufficient distinct usable cycle_code values for a cross-period comparison.")


Status: BLOCKED_MISSING_REGRESSION_DATA
Regression data folder: data/raw/regression
Required official files: Train.csv, Test.csv, SampleSubmission.csv
Place the official regression Train.csv, Test.csv, and SampleSubmission.csv in data/raw/regression/ (or TP2_DATA_DIR/regression/), then rerun.


## Results

Angka, tabel, dan grafik muncul setelah cell analisis berjalan. Jika status `BLOCKED`, temuan relasi belum tersedia. Laporan agregat disimpan di `reports/eda/samuel_regression_eda.json`; grafik yang dibuat tersimpan di direktori yang sama.

### 1. Overall and period association

In [2]:

if report["status"] == "COMPLETE":
    rows = [{"group": "all", **report["relationship"]["overall"]}]
    rows.extend({"group": row["period"], **row} for row in report["relationship"]["by_period"])
    display(pd.DataFrame(rows)[["group", "n_complete", "pearson_r", "spearman_rho"]])
    print("Exact repeated impact_load check:")
    display(pd.DataFrame([report["relationship"]["exact_impact_repeat_audit"]]))
    print("Per-cycle correlations:")
    if report["relationship"]["by_cycle"]:
        display(pd.DataFrame(report["relationship"]["by_cycle"])[
            ["cycle_code", "n_complete", "pearson_r", "spearman_rho", "target_nonmissing_n"]])
    else:
        print("No usable cycle_code groups are available.")
else:
    print("No target-feature association can be reported until the official regression data is available.")


No target-feature association can be reported until the official regression data is available.


### 2. Relationship shape and conditional spread

In [3]:

if report["status"] == "COMPLETE":
    figure = PROJECT_ROOT / "reports" / "eda" / "samuel_impact_threat_hexbin.png"
    if figure.is_file():
        display(Image(filename=str(figure)))


In [4]:

if report["status"] == "COMPLETE":
    display(pd.DataFrame(report["relationship"]["by_period_and_impact_bin"]))
    figure = PROJECT_ROOT / "reports" / "eda" / "samuel_impact_bins_by_period.png"
    if figure.is_file():
        display(Image(filename=str(figure)))


### 3. Target distribution by period

In [5]:

if report["status"] == "COMPLETE":
    display(pd.DataFrame(report["temporal"]["period_target_distribution"]))
    figure = PROJECT_ROOT / "reports" / "eda" / "samuel_threat_by_period.png"
    if figure.is_file():
        display(Image(filename=str(figure)))


## Takeaways

Tafsirkan arah dan kekuatan korelasi bersama bentuk sebaran serta variasi `threat_rank` pada nilai `impact_load` serupa. Bandingkan periode dengan batas bin `impact_load` yang sama dan periksa ukuran kelompok. Perbedaan deskriptif hanya menjadi indikasi untuk pemeriksaan lanjutan; jangan menyebutnya drift atau efek fitur tanpa bukti tambahan.